# Comparar runs e escolher candidato

POC DEMO — contrato PROVISÓRIO / PENDENTE DE HOMOLOGAÇÃO. Execute em kernel novo. Credenciais somente por ambiente.


Configuração tipada. Pré-requisitos: Docker saudável, seed DEMO, token por ambiente. Entradas: parâmetros abaixo; saídas: tabelas, IDs e relatório JSON/Markdown. Escritas exigem confirmação EXECUTE_DEMO e chave estável; repetir chave igual recupera o mesmo job. Espere READY; erros de dependência exigem verificar docker compose logs.

In [1]:
import os, json
import pandas as pd
from IPython.display import display, Markdown
from src.workbench.journey_config import Settings
cfg=Settings.load()
from src.workbench.notebook_config import Parameters
from src.workbench.operations_sdk import call, report, await_job
params=Parameters.load()
display(params.model_dump(mode="json"))
from src.workbench.manual_validation import display_locations, audit, export_session
display({"state":"READY", "write_enabled":params.execute_write, "tracking":"remote Docker service", "browser_ui":cfg.ui})
evidence={}
NOTEBOOK_ID="06_avaliacao_comparativa"


{'mode': 'READ_EXISTING',
 'proposal_id': None,
 'snapshot_id': 'fb931e1e-43d0-5ff8-938c-0d2ba4a68794',
 'model_name': 'planilha-turmas-demo',
 'model_version': '',
 'registry_prefix': '',
 'registry_page_token': '',
 'campaign_id': None,
 'approval_id': None,
 'execute_write': False,
 'confirmation': '',
 'request_key': '',
 'requester': 'scientist-demo',
 'algorithm': 'ridge',
 'alpha': 1.0,
 'capacity': 40,
 'model_count': 3,
 'campaign_display_name': 'POC DEMO • Validação manual',
 'parallel_jobs': 2,
 'campaign_action': 'observe',
 'governance_action': 'read',
 'review_action': 'APPROVE',
 'reason': '',
 'lineage_kind': 'proposal',
 'external_id': '',
 'reset_mode': 'preview',
 'reinitialize': 'demo-ready',
 'reset_plan_id': None}

{'state': 'READY',
 'write_enabled': False,
 'tracking': 'remote Docker service',
 'browser_ui': 'http://127.0.0.1:5000'}

1. Filtrar por snapshot explícito. Compare apenas a mesma janela; baseline e métricas são demonstrativos.

In [2]:
from src.workbench.operations_sdk import comparison
records=comparison(cfg,str(params.snapshot_id) if params.snapshot_id else "",limit=500)
table=pd.DataFrame(records);display(table)
evidence["runs"]=records


,run_id,snapshot_id,model_key,algorithm,train_rows,evaluation_rows,url,target,evaluation_window,dataset_sha256,mae,rmse,training_seconds,inference_rows,inference_seconds,baseline_mae,baseline_rmse,duration_seconds,cpu_seconds,peak_rss_kib
0,3181e902380e45eaba74747dc1153aba,fb931e1e-43d0-5ff8-938c-0d2ba4a68794,planilha-turmas-demo,Ridge,None,None,http://127.0.0.1:5000/#/experiments/2/runs/318...,students,None,5fc39eeeead59aa48813182e6a93d2308a4a9243655501...,3.758631,4.914227,0.010864,90.0,0.197649,NaN,NaN,NaN,NaN,NaN
1,1932809c8fa2454a9d6a86920ea9e707,fb931e1e-43d0-5ff8-938c-0d2ba4a68794,planilha-turmas-demo,Ridge,None,None,http://127.0.0.1:5000/#/experiments/2/runs/193...,students,None,5fc39eeeead59aa48813182e6a93d2308a4a9243655501...,3.758631,4.914227,0.009328,90.0,0.457377,NaN,NaN,NaN,NaN,NaN
2,2e3be451004e4dfe91ccca53258d8fdb,fb931e1e-43d0-5ff8-938c-0d2ba4a68794,hi.pt.demanda.DEMO-C3.DEMO-CR3.P1,ridge,33,3,http://127.0.0.1:5000/#/experiments/3/runs/2e3...,students,4051..4051,eb3c6fe1c9fae5cad99eb26311d84b2cdfb2ce77154e82...,9.832086,9.832086,NaN,NaN,NaN,9.090909,9.090909,0.522162,0.119638,271496.0
3,e767ef652f9944c99c802bbe5439b24d,fb931e1e-43d0-5ff8-938c-0d2ba4a68794,hi.pt.demanda.DEMO-C1.DEMO-CR3.P1,ridge,33,3,http://127.0.0.1:5000/#/experiments/3/runs/e76...,students,4051..4051,b672ce6e1266567d8ca2af160fe019fd628731deea138a...,11.992513,11.992513,NaN,NaN,NaN,4.545455,4.545455,0.535846,0.155852,269976.0
4,24571ecb25404ceab73644ba7df4cd0e,fb931e1e-43d0-5ff8-938c-0d2ba4a68794,hi.pt.demanda.DEMO-C1.DEMO-CR5.P1,ridge,33,3,http://127.0.0.1:5000/#/experiments/3/runs/245...,students,4051..4051,70d59a1454402057359cc34a1ab23e828a575c6833d509...,3.899465,3.899465,NaN,NaN,NaN,7.181818,7.181818,0.829515,0.171083,271496.0
5,a80447d4938943a3bd2729fa0dc13ca6,fb931e1e-43d0-5ff8-938c-0d2ba4a68794,hi.pt.demanda.DEMO-C3.DEMO-CR5.P1,ridge,33,3,http://127.0.0.1:5000/#/experiments/3/runs/a80...,students,4051..4051,9569251dbc18ce805d99cdeb8b3fd2b6bb51bbe44209bb...,10.068449,10.068449,NaN,NaN,NaN,2.727273,2.727273,0.511086,0.132823,269976.0
6,7aaa24878c174cb99d6c844030839efd,fb931e1e-43d0-5ff8-938c-0d2ba4a68794,hi.pt.demanda.DEMO-C2.DEMO-CR1.P1,ridge,33,3,http://127.0.0.1:5000/#/experiments/3/runs/7aa...,students,4051..4051,4df818e76724c968015967b2e682cb2756dc1b00c1526c...,0.060963,0.060963,NaN,NaN,NaN,3.272727,3.272727,0.698760,0.142162,271112.0
7,39af1bfa43b843c2aaf0820f13734ba2,fb931e1e-43d0-5ff8-938c-0d2ba4a68794,hi.pt.demanda.DEMO-C2.DEMO-CR4.P1,ridge,33,3,http://127.0.0.1:5000/#/experiments/3/runs/39a...,students,4051..4051,9cab8d1694a37bf6dc42de08dccd0c3836e132060f5bdf...,0.389305,0.389305,NaN,NaN,NaN,0.636364,0.636364,0.557712,0.134141,269976.0
8,581d5e9c9e7d4b76bd7cb257d5c0ea4f,fb931e1e-43d0-5ff8-938c-0d2ba4a68794,hi.pt.demanda.DEMO-C3.DEMO-CR2.P2,ridge,33,3,http://127.0.0.1:5000/#/experiments/3/runs/581...,students,4051..4051,b82d481eac578e20c4709341ef155febc3d982aa6a8c52...,13.834225,13.834225,NaN,NaN,NaN,8.363636,8.363636,0.632570,0.132796,270920.0
9,a7c6c5c955ca42b0af3b4645e89047b8,fb931e1e-43d0-5ff8-938c-0d2ba4a68794,hi.pt.demanda.DEMO-C3.DEMO-CR2.P1,ridge,33,3,http://127.0.0.1:5000/#/experiments/3/runs/a7c...,students,4051..4051,1ef3239d71ac5354a80f90e58a96fa7a02384d54f0bfe2...,1.385027,1.385027,NaN,NaN,NaN,5.090909,5.090909,0.899669,0.442683,257116.0


2. Eleger candidato por menor MAE dentro de um único snapshot/janela, sem aprovar ou promover.

In [3]:
if not table.empty and "mae" in table and params.snapshot_id:
    comparable=table.dropna(subset=["snapshot_id","model_key","evaluation_window","target","dataset_sha256"])
    groups=comparable.groupby(["snapshot_id","model_key","evaluation_window","target","dataset_sha256"])
    candidates=[g.sort_values("mae").iloc[0].to_dict() for _,g in groups if g.mae.notna().any()]
    display(pd.DataFrame(candidates));evidence["candidates"]=candidates
else:
    display({"candidate":"NOT_EVALUATED","reason":"Selecione snapshot e runs com mesma janela"})
evidence["promotion"]="NOT_EXECUTED"


,run_id,snapshot_id,model_key,algorithm,train_rows,evaluation_rows,url,target,evaluation_window,dataset_sha256,mae,rmse,training_seconds,inference_rows,inference_seconds,baseline_mae,baseline_rmse,duration_seconds,cpu_seconds,peak_rss_kib
0,466af8046be248029c27e23cce2ee70e,fb931e1e-43d0-5ff8-938c-0d2ba4a68794,hi.demo.integration.70491a51,ridge,960,90,http://127.0.0.1:5000/#/experiments/3/runs/466...,students,4051..4051,5fc39eeeead59aa48813182e6a93d2308a4a9243655501...,3.758631,4.914227,NaN,NaN,NaN,7.466667,8.821497,0.422288,1.441713,274844.0
1,420b1e103c8741ddbb8ec66427a95c0b,fb931e1e-43d0-5ff8-938c-0d2ba4a68794,hi.demo.notebook.a9291c01,ridge,960,90,http://127.0.0.1:5000/#/experiments/3/runs/420...,students,4051..4051,5fc39eeeead59aa48813182e6a93d2308a4a9243655501...,3.758631,4.914227,NaN,NaN,NaN,7.466667,8.821497,0.570752,0.225850,276956.0
2,d6023330e3944990ac27ecf9fee47e51,fb931e1e-43d0-5ff8-938c-0d2ba4a68794,hi.demo.probe.18cb82de,ridge,960,90,http://127.0.0.1:5000/#/experiments/3/runs/d60...,students,4051..4051,5fc39eeeead59aa48813182e6a93d2308a4a9243655501...,3.758631,4.914227,NaN,NaN,NaN,7.466667,8.821497,0.786253,1.831697,260124.0
3,8de6f68bbf5c41b3a67a461e3d67775a,fb931e1e-43d0-5ff8-938c-0d2ba4a68794,hi.demo.v3-negative.699bb4f2,ridge,960,90,http://127.0.0.1:5000/#/experiments/3/runs/8de...,students,4051..4051,5fc39eeeead59aa48813182e6a93d2308a4a9243655501...,3.758631,4.914227,NaN,NaN,NaN,7.466667,8.821497,0.484472,0.197892,275804.0
4,f6f552029db24af6bd4a1385b726017d,fb931e1e-43d0-5ff8-938c-0d2ba4a68794,hi.pt.demanda.DEMO-C1.DEMO-CR1.P1,ridge,33,3,http://127.0.0.1:5000/#/experiments/3/runs/f6f...,students,4051..4051,22b3c1491ef5f01f1c646bc43fd8350d324bce87b3bf4e...,2.451818,2.451818,NaN,NaN,NaN,2.181818,2.181818,0.417910,0.128668,267356.0
5,2e6951af59584fc59e50376ff4c3dc99,fb931e1e-43d0-5ff8-938c-0d2ba4a68794,hi.pt.demanda.DEMO-C1.DEMO-CR1.P2,ridge,33,3,http://127.0.0.1:5000/#/experiments/3/runs/2e6...,students,4051..4051,7d0b348d3136b500c89edb327d0976d12a8455c33d770b...,6.622460,6.622460,NaN,NaN,NaN,4.363636,4.363636,0.617908,0.141726,277532.0
6,62f5daa090c44ec096d0af03840d0269,fb931e1e-43d0-5ff8-938c-0d2ba4a68794,hi.pt.demanda.DEMO-C1.DEMO-CR2.P1,ridge,33,3,http://127.0.0.1:5000/#/experiments/3/runs/62f...,students,4051..4051,7900273a22a096bef7a848c3bd0671e7d89290caec9f9d...,8.291979,8.291979,NaN,NaN,NaN,10.727273,10.727273,0.554005,0.122885,270152.0
7,a80e9b36e4764129b935ff4ff804fc71,fb931e1e-43d0-5ff8-938c-0d2ba4a68794,hi.pt.demanda.DEMO-C1.DEMO-CR2.P2,ridge,33,3,http://127.0.0.1:5000/#/experiments/3/runs/a80...,students,4051..4051,66d2a715916fe4ed3213f60d31b28918a301a8402f9857...,0.313369,0.313369,NaN,NaN,NaN,0.454545,0.454545,0.695868,0.148878,269000.0
8,e767ef652f9944c99c802bbe5439b24d,fb931e1e-43d0-5ff8-938c-0d2ba4a68794,hi.pt.demanda.DEMO-C1.DEMO-CR3.P1,ridge,33,3,http://127.0.0.1:5000/#/experiments/3/runs/e76...,students,4051..4051,b672ce6e1266567d8ca2af160fe019fd628731deea138a...,11.992513,11.992513,NaN,NaN,NaN,4.545455,4.545455,0.535846,0.155852,269976.0
9,4720a0478b5e4fb8a222f34cec636060,fb931e1e-43d0-5ff8-938c-0d2ba4a68794,hi.pt.demanda.DEMO-C1.DEMO-CR3.P2,ridge,33,3,http://127.0.0.1:5000/#/experiments/3/runs/472...,students,4051..4051,37bfddcc1aaf0536473fb1e13fae62d5176f140a87862d...,0.567914,0.567914,NaN,NaN,NaN,4.909091,4.909091,0.635823,0.152295,278492.0


Guardar evidência desta execução e links. Esperado: caminho do relatório, sem credenciais.

In [4]:
report_status="PARTIAL" if any(isinstance(value,str) and value.startswith("NOT_EXECUTED") for value in evidence.values()) else "PASS"
if evidence.get("campaign",{}).get("status") in ("FAILED","PARTIAL","CANCELLED","PAUSED"):
    report_status="PARTIAL"
display(report(NOTEBOOK_ID,evidence,status=report_status,generation="v3",limitations=["Dados DEMO e contrato provisório; regras não avaliadas não são homologação", "Execução sem erro do notebook não implica execução de operações opt-in desativadas"]))


'/reports/v3/06_avaliacao_comparativa.json'